In [3]:
# 1. Clone repository & system dependencies
%cd /content
!rm -rf /content/subtitlesssss
!git clone https://github.com/aananyaph/subtitlesssss
%cd /content/subtitlesssss

# Compatibility edits from original repository
!sed -i 's/^from typing import Optional$/from typing import Any, Callable, Dict, Optional/' main_app.py
!sed -i '0,/^import logging$/s//from __future__ import annotations\nimport logging/' app/loopback_capture.py

# Ensure system GUI/font libraries for offscreen Qt are installed
!apt-get install -y --no-install-recommends libgl1 libegl1 libxkbcommon0 libdbus-1-3 libfontconfig1
print("✅ System dependencies and repository ready.")


/content
Cloning into 'subtitlesssss'...
remote: Enumerating objects: 267, done.
remote: Counting objects: 100% (267/267), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 267 (delta 106), reused 238 (delta 85), pack-reused 0 (from 0)
Receiving objects: 100% (267/267), 5.23 MiB | 27.72 MiB/s, done.
Resolving deltas: 100% (106/106), done.
/content/subtitlesssss
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libgl1 is already the newest version (1.7.0-1build1).
libegl1 is already the newest version (1.7.0-1build1).
libxkbcommon0 is already the newest version (1.6.0-1build1).
libdbus-1-3 is already the newest version (1.14.10-4ubuntu4.1).
libfontconfig1 is already the newest version (2.15.0-1.1ubuntu2).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
✅ System dependencies and repository ready.


In [4]:
# 2. Install required IndicTrans2, ONNX-GPU, and benchmarking packages
# Pre-install cython and wheel for IndicTransToolkit
!pip install --no-cache-dir wheel cython

# Install backend packages (note: faster-whisper pulls in CPU onnxruntime by default)
!pip install faster-whisper websockets jiwer scipy PyQt6 \
    sentencepiece sacremoses sacrebleu tokenizers indic-nlp-library pyngrok indictranstoolkit

# CRITICAL: faster-whisper installs CPU onnxruntime. We must swap it for onnxruntime-gpu:
!pip uninstall -y onnxruntime onnxruntime-gpu
!pip install onnxruntime-gpu
print("✅ All Python packages and onnxruntime-gpu successfully installed.")


  Using cached onnxruntime-1.31.1-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.8 kB)
Using cached onnxruntime-1.31.1-cp313-cp313-manylinux_2_28_x86_64.whl (23.8 MB)
Found existing installation: onnxruntime 1.31.1
Uninstalling onnxruntime-1.31.1:
  Successfully uninstalled onnxruntime-1.31.1
Found existing installation: onnxruntime-gpu 1.31.1
Uninstalling onnxruntime-gpu-1.31.1:
  Successfully uninstalled onnxruntime-gpu-1.31.1
  Using cached onnxruntime_gpu-1.31.1-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (6.2 kB)
Using cached onnxruntime_gpu-1.31.1-cp313-cp313-manylinux_2_28_x86_64.whl (255.6 MB)
✅ All Python packages and onnxruntime-gpu successfully installed.


In [13]:
# 3. Verify CUDA, PyTorch, and ONNX Runtime GPU Providers
!nvidia-smi

import os
import sys
import types
import torch
from google.colab import userdata

# Compatibility fixes for Hugging Face transformers >= 4.45
# 1. Hugging Face removed transformers.onnx in newer releases, but IndicTrans2
#    remote code (configuration_indictrans.py) imports OnnxConfig, OnnxSeq2SeqConfigWithPast,
#    and compute_effective_axis_dimension from transformers.onnx and transformers.onnx.utils.
if 'transformers.onnx' not in sys.modules or not hasattr(sys.modules.get('transformers.onnx'), '__path__'):
    _mock_onnx = sys.modules.get('transformers.onnx') or types.ModuleType('transformers.onnx')
    _mock_onnx.__path__ = []
    class _OnnxConfig:
        default_fixed_batch = 2
        default_fixed_sequence = 16
    class _OnnxSeq2SeqConfigWithPast(_OnnxConfig): pass
    _mock_onnx.OnnxConfig = _OnnxConfig
    _mock_onnx.OnnxSeq2SeqConfigWithPast = _OnnxSeq2SeqConfigWithPast
    sys.modules['transformers.onnx'] = _mock_onnx

if 'transformers.onnx.utils' not in sys.modules:
    _mock_onnx_utils = types.ModuleType('transformers.onnx.utils')
    _mock_onnx_utils.compute_effective_axis_dimension = lambda dimension, fixed_dimension=None, num_token_to_add=0: ((fixed_dimension or 1) + (num_token_to_add or 0)) if (dimension == -1 or dimension is None) else dimension
    sys.modules['transformers.onnx.utils'] = _mock_onnx_utils
    setattr(sys.modules['transformers.onnx'], 'utils', _mock_onnx_utils)

try:
    import transformers
    setattr(transformers, 'onnx', sys.modules['transformers.onnx'])
except Exception:
    pass

# 2. IndicTransToolkit expects PreTrainedTokenizerBase in tokenization_utils
try:
    import transformers.tokenization_utils as _tu
    import transformers.tokenization_utils_base as _tub
    if not hasattr(_tu, 'PreTrainedTokenizerBase'):
        _tu.PreTrainedTokenizerBase = _tub.PreTrainedTokenizerBase
except Exception:
    pass

# 3. IndicTransTokenizer assigns special tokens before super().__init__(),
# causing AttributeError on _special_tokens_map in transformers >= 4.45.
try:
    from transformers.tokenization_utils_base import PreTrainedTokenizerBase
    if not getattr(PreTrainedTokenizerBase, "_patched_special_tokens", False):
        _orig_setattr = PreTrainedTokenizerBase.__setattr__
        def _safe_setattr(self, key, value):
            key_clean = key.removesuffix("_ids").removesuffix("_id") if key.endswith(("_id", "_ids")) else key
            if key_clean in getattr(self, "SPECIAL_TOKENS_ATTRIBUTES", ()):
                if "_special_tokens_map" not in self.__dict__:
                    self.__dict__["_special_tokens_map"] = {}
            return _orig_setattr(self, key, value)
        PreTrainedTokenizerBase.__setattr__ = _safe_setattr
        PreTrainedTokenizerBase._patched_special_tokens = True
except Exception:
    pass

# Hugging Face token from Colab secrets
try:
    _hf_token = userdata.get('HF_TOKEN')
    if _hf_token:
        os.environ["HF_TOKEN"] = _hf_token.strip()
        print("✅ Successfully loaded HF_TOKEN from Colab Secrets.")
    else:
        print("⚠️ HF_TOKEN not found in Colab Secrets. Please make sure to add it under the '🔑' tab with the name 'HF_TOKEN'.")
except Exception as e:
    print(f"Could not load HF_TOKEN from secrets: {e}")

# Assert CUDA availability for PyTorch
print("PyTorch version:", torch.__version__)
assert torch.cuda.is_available(), "CUDA is unavailable in PyTorch! Switch to T4 GPU."
print("✅ PyTorch CUDA GPU:", torch.cuda.get_device_name(0))

import onnxruntime as ort
print("ONNX Runtime available providers:", ort.get_available_providers())
assert "CUDAExecutionProvider" in ort.get_available_providers(), "CUDAExecutionProvider unavailable!"
print("✅ ONNX Runtime CUDA provider verified.")

from IndicTransToolkit import IndicProcessor
ip = IndicProcessor(inference=True)
print("✅ IndicProcessor successfully initialized.")

from transformers import AutoTokenizer
print("Verifying IndicTrans2 tokenizer load...")
_tok = AutoTokenizer.from_pretrained("ai4bharat/indictrans2-en-indic-1B", trust_remote_code=True, token=os.environ.get("HF_TOKEN"))
print("✅ IndicTrans2 Tokenizer verified:", type(_tok).__name__)


In [15]:
# 5. Pre-download IndicTrans2 1B repos and IndicConformer
import os, shutil
from huggingface_hub import snapshot_download

hf_token = os.environ.get("HF_TOKEN") or None

# Pre-download IndicConformer for Indic ASR
print("Downloading IndicConformer ASR model...")
snapshot_download("ai4bharat/indic-conformer-600m-multilingual", token=hf_token)

root = "/root/.cache/huggingface/hub/models--ai4bharat--indic-conformer-600m-multilingual/snapshots"
if os.path.exists(root):
    copied = 0
    for dp, _, filenames in os.walk(root):
        for filename in filenames:
            file_path = os.path.join(dp, filename)
            if os.path.islink(file_path):
                target = os.path.realpath(file_path)
                if not os.path.exists(target):
                    raise FileNotFoundError(f"Broken symlink: {file_path} -> {target}")
                os.remove(file_path)
                shutil.copy2(target, file_path)
                copied += 1
    print("Replaced symlinks with file copies for IndicConformer:", copied)

# Pre-download IndicTrans2 1B translation models
repos_1b = [
    "ai4bharat/indictrans2-en-indic-1B",
    "ai4bharat/indictrans2-indic-en-1B",
    "ai4bharat/indictrans2-indic-indic-1B",
]

for repo in repos_1b:
    try:
        print(f"Pre-downloading/verifying: {repo}...")
        snapshot_download(repo, token=hf_token)
        print(f"✅ Ready: {repo}")
    except Exception as e:
        print(f"⚠️ Could not download {repo}: {e}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Replaced symlinks with file copies for IndicConformer: 0
Pre-downloading/verifying: ai4bharat/indictrans2-en-indic-1B...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Ready: ai4bharat/indictrans2-en-indic-1B
Pre-downloading/verifying: ai4bharat/indictrans2-indic-en-1B...


... (2 outputs left out)

✅ Ready: ai4bharat/indictrans2-indic-en-1B
Pre-downloading/verifying: ai4bharat/indictrans2-indic-indic-1B...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Ready: ai4bharat/indictrans2-indic-indic-1B


In [2]:
# 6. Apply dynamic ASR config patch and extract ONNX fallback translator
import os

path = "/content/subtitlesssss/backend/app_bridge.py"
with open(path, "r", encoding="utf-8") as f:
    s = f.read()

if "def apply_asr_config" not in s:
    helper = '''# ---------- Dynamic ASR config (from client HANDSHAKE) ----------
_current_cfg_key = None

def apply_asr_config(cfg):
    """Rebuild pipeline to match client's audio/subtitle language choice."""
    global pipeline, _current_cfg_key
    key = (
        cfg.get("model_name") or "base",
        cfg.get("indic_lang") or None,
        cfg.get("asr_language") or None,
        cfg.get("asr_task") or None,
    )
    if pipeline is not None and key == _current_cfg_key:
        logging.info(f"[Bridge] ASR config unchanged {key}; keeping pipeline")
        return
    logging.info(f"[Bridge] Switching ASR config -> model={key[0]} indic={key[1]} lang={key[2]} task={key[3]}")
    import main_app
    old = pipeline
    if old is not None:
        try:
            old.stop()
        except Exception as e:
            logging.warning(f"[Bridge] old pipeline stop failed: {e}")
    if key[1]:
        from indic_backend import load_indic_model
        load_indic_model()
    new = main_app.DesktopLiveSubtitlePipeline(
        overlay=BridgeOverlay(),
        model_name=key[0],
        indic_lang=key[1],
        asr_language=key[2],
        asr_task=key[3],
    )
    try:
        new.start()
    except Exception as e:
        logging.warning(f"[Bridge] new pipeline start failed (no audio device?) {e}")
    pipeline = new
    _current_cfg_key = key
    logging.info("[Bridge] ASR config applied")

'''
    marker = "# ---------- WebSocket handler ----------"
    s = s.replace(marker, helper + marker)
    old = '                    logging.info(f"[Bridge] HANDSHAKE {data}")\n'
    new = '''                    logging.info(f"[Bridge] HANDSHAKE {data}")
                    _cfg = data.get("asrConfig")
                    if _cfg:
                        try:
                            await asyncio.get_running_loop().run_in_executor(None, apply_asr_config, _cfg)
                        except Exception as e:
                            logging.error(f"[Bridge] apply_asr_config failed: {e}", exc_info=True)
                            await ws.send(json.dumps({"type": "final", "text": f"Server model load failed: {e}", "is_final": True}))
'''
    s = s.replace(old, new)
    with open(path, "w", encoding="utf-8") as f:
        f.write(s)
    print("app_bridge.py patched: dynamic ASR config on HANDSHAKE")

# Patch main_app.py for Kannada unicode
path_main = "/content/subtitlesssss/main_app.py"
with open(path_main, "r", encoding="utf-8") as f:
    s_main = f.read()

if "_kn_fix" not in s_main:
    old_m = '                text = " ".join(s["text"] for s in result.get("segments", [])).strip()\n'
    new_m = '''                text = " ".join(s["text"] for s in result.get("segments", [])).strip()
                if self.indic_lang == "kn" and text:
                    # _kn_fix: stray Tamil block (U+0B80-0BFF) -> Kannada block (U+0C80-0CFF)
                    text = "".join(
                        chr(ord(c) - 0x0B80 + 0x0C80) if 0x0B80 <= ord(c) <= 0x0BFF else c
                        for c in text
                    )
'''
    s_main = s_main.replace(old_m, new_m)
    with open(path_main, "w", encoding="utf-8") as f:
        f.write(s_main)
    print("main_app.py patched: kn script sanitize in Indic path")

# Extract translator_live.py (ONNX fallback) if needed
import ast
root = "/content/subtitlesssss"
if not os.path.exists(f"{root}/translator_live.py"):
    src = open(f"{root}/pipeline.py", encoding="utf-8").read()
    lines = src.splitlines(keepends=True)
    chunks = []
    for node in ast.parse(src).body:
        if isinstance(node, ast.Assign) and any(getattr(t, "id", None) == "LANG_MAP" for t in node.targets):
            chunks.append("".join(lines[node.lineno - 1:node.end_lineno]))
        elif isinstance(node, ast.ClassDef) and node.name == "IndicTransONNXModel":
            chunks.append("".join(lines[node.lineno - 1:node.end_lineno]))
    header = '''import json, logging
from pathlib import Path
import numpy as np
import onnxruntime as ort
from huggingface_hub import snapshot_download
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from indicnlp.tokenize import indic_detokenize
from indicnlp.transliterate.unicode_transliterate import UnicodeIndicTransliterator
from sacremoses import MosesDetokenizer, MosesTokenizer
from tokenizers import Tokenizer

logger = logging.getLogger("translator_live")

'''
    open(f"{root}/translator_live.py", "w", encoding="utf-8").write(header + "".join(chunks))
    p = f"{root}/translator_live.py"
    t = open(p, encoding="utf-8").read()
    old_l = "        self.snap = Path(snapshot_download(repo_id))\n"
    new_l = old_l + '''        import os, shutil  # _symlink_fix
        for _dp, _, _fns in os.walk(self.snap):
            for _fn in _fns:
                _fp = os.path.join(_dp, _fn)
                if os.path.islink(_fp):
                    _t = os.path.realpath(_fp)
                    os.remove(_fp)
                    shutil.copy2(_t, _fp)
'''
    if "_symlink_fix" not in t and old_l in t:
        open(p, "w", encoding="utf-8").write(t.replace(old_l, new_l))
    print("translator_live.py ready")

In [17]:
# 7. STEP 4: Direct Model Benchmark (6 Directions, 20 Sentences Each)
!cd /content/subtitlesssss && git pull origin main

import importlib
import gc
import json
import os
import sys
import time
import numpy as np

sys.path.insert(0, "/content/subtitlesssss")

if 'translator_gpu' in sys.modules:
    importlib.reload(sys.modules['translator_gpu'])
if 'translation_proxy' in sys.modules:
    importlib.reload(sys.modules['translation_proxy'])

import sacrebleu
import torch
import translation_proxy as tp

# 20 parallel test sentences across English, Hindi, and Kannada
test_sentences = {
    "en": [
        "Hello, how are you today?",
        "Good morning to everyone present here.",
        "The weather is very pleasant this evening.",
        "Where is the nearest railway station?",
        "Can you please help me with this problem?",
        "We are developing a real-time subtitle translation system.",
        "Artificial intelligence is transforming communication across the world.",
        "I enjoy listening to classical Indian music while working.",
        "Please make sure to submit your assignment before Friday.",
        "The train will arrive at platform number three in ten minutes.",
        "India is a country with rich cultural heritage and diversity.",
        "What time does the library close on weekends?",
        "Health and education are fundamental rights for every citizen.",
        "She works as a software engineer in Bengaluru.",
        "Can you recommend a good book to read?",
        "Water conservation is essential for our future generations.",
        "The children are playing happily in the garden.",
        "Scientists have discovered a new species in the Western Ghats.",
        "Learning multiple languages expands cognitive abilities and empathy.",
        "Thank you very much for your time and guidance."
    ],
    "hi": [
        "नमस्ते, आप आज कैसे हैं?",
        "यहाँ उपस्थित सभी लोगों को सुप्रभात।",
        "आज शाम का मौसम बहुत सुहावना है।",
        "निकटतम रेलवे स्टेशन कहाँ है?",
        "क्या आप इस समस्या में मेरी मदद कर सकते हैं?",
        "हम एक वास्तविक समय उपशीर्षक अनुवाद प्रणाली विकसित कर रहे हैं।",
        "कृत्रिम बुद्धिमत्ता दुनिया भर में संचार को बदल रही है।",
        "मुझे काम करते समय भारतीय शास्त्रीय संगीत सुनना पसंद है।",
        "कृपया शुक्रवार से पहले अपना असाइनमेंट जमा करना सुनिश्चित करें।",
        "ट्रेन दस मिनट में प्लेटफॉर्म नंबर तीन पर पहुंचेगी।",
        "भारत समृद्ध सांस्कृतिक विरासत और विविधता वाला देश है।",
        "सप्ताहांत पर पुस्तकालय किस समय बंद होता है?",
        "स्वास्थ्य और शिक्षा प्रत्येक नागरिक का मौलिक अधिकार है।",
        "वह बेंगलुरु में एक सॉफ्टवेयर इंजीनियर के रूप में काम करती है।",
        "क्या आप पढ़ने के लिए किसी अच्छी किताब की सिफारिश कर सकते हैं?",
        "जल संरक्षण हमारी भावी पीढ़ियों के लिए आवश्यक है।",
        "बच्चे बगीचे में खुशी से खेल रहे हैं।",
        "वैज्ञानिकों ने पश्चिमी घाट में एक नई प्रजाति की खोज की है।",
        "कई भाषाएं सीखने से सोचने-समझने की क्षमता और संवेदनशीलता बढ़ती है।",
        "आपके समय और मार्गदर्शन के लिए बहुत-बहुत धन्यवाद।"
    ],
    "kn": [
        "ನಮಸ್ಕಾರ, ನೀವು ಇಂದು ಹೇಗಿದ್ದೀರಿ?",
        "ಇಲ್ಲಿ ನೆರೆದಿರುವ ಎಲ್ಲರಿಗೂ ಶುಭೋದಯ.",
        "ಇಂದು ಸಂಜೆಯ ವಾತಾವರಣ ತುಂಬಾ ಆಹ್ಲಾದಕರವಾಗಿದೆ.",
        "ಹತ್ತಿರದ ರೈಲ್ವೆ ನಿಲ್ದಾಣ ಎಲ್ಲಿದೆ?",
        "ದಯವಿಟ್ಟು ಈ ಸಮಸ್ಯೆಯಲ್ಲಿ ನನಗೆ ಸಹಾಯ ಮಾಡಬಹುದೇ?",
        "ನಾವು ನೈಜ ಸಮಯದ ಉಪಶೀರ್ಷಿಕೆ ಅನುವಾದ ವ್ಯವಸ್ಥೆಯನ್ನು ಅಭಿವೃದ್ಧಿಪಡಿಸುತ್ತಿದ್ದೇವೆ.",
        "ಕೃತಕ ಬುದ್ಧಿಮತ್ತೆಯು ಪ್ರಪಂಚದಾದ್ಯಂತ ಸಂವಹನವನ್ನು ಬದಲಾಯಿಸುತ್ತಿದೆ.",
        "ಕೆಲಸ ಮಾಡುವಾಗ ಭಾರತೀಯ ಶಾಸ್ತ್ರೀಯ ಸಂಗೀತವನ್ನು ಕೇಳಲು ನಾನು ಇಷ್ಟಪಡುತ್ತೇನೆ.",
        "ದಯವಿಟ್ಟು ಶುಕ್ರವಾರದ ಮೊದಲು ನಿಮ್ಮ ನಿಯೋಜನೆಯನ್ನು ಸಲ್ಲಿಸಲು ಖಚಿತಪಡಿಸಿಕೊಳ್ಳಿ.",
        "ರೈಲು ಹತ್ತು ನಿಮಿಷಗಳಲ್ಲಿ ಪ್ಲಾಟ್‌ಫಾರ್ಮ್ ಸಂಖ್ಯೆ ಮೂರಕ್ಕೆ ಬರಲಿದೆ.",
        "ಭಾರತವು ಶ್ರೀಮಂತ ಸಾಂಸ್ಕೃತಿಕ ಪರಂಪರೆ ಮತ್ತು ವೈವಿಧ್ಯತೆಯನ್ನು ಹೊಂದಿರುವ ದೇಶವಾಗಿದೆ.",
        "ವಾರಾಂತ್ಯದಲ್ಲಿ ಗ್ರಂಥಾಲಯವು ಯಾವ ಸಮಯಕ್ಕೆ ಮುಚ್ಚುತ್ತದೆ?",
        "ಆರೋಗ್ಯ ಮತ್ತು ಶಿಕ್ಷಣ ಪ್ರತಿಯೊಬ್ಬ ನಾಗರಿಕನ ಮೂಲಭೂತ ಹಕ್ಕು.",
        "ಅವರು ಬೆಂಗಳೂರಿನಲ್ಲಿ ಸಾಫ್ಟ್‌ವೇರ್ ಎಂಜಿನಿಯರ್ ಆಗಿ ಕೆಲಸ ಮಾಡುತ್ತಿದ್ದಾರೆ.",
        "ಓದಲು ಯಾವುದಾದರೂ ಒಳ್ಳೆಯ ಪುಸ್ತಕವನ್ನು ನೀವು ಶಿಫಾರಸು ಮಾಡಬಹುದೇ?",
        "ನಮ್ಮ ಮುಂದಿನ ಪೀಳಿಗೆಗೆ ಜಲ ಸಂರಕ್ಷಣೆ ಅತ್ಯಗತ್ಯ.",
        "ಮಕ್ಕಳು ತೋಟದಲ್ಲಿ ಸಂತೋಷದಿಂದ ಆಟವಾಡುತ್ತಿದ್ದಾರೆ.",
        "ವಿಜ್ಞಾನಿಗಳು ಪಶ್ಚಿಮ ಘಟ್ಟಗಳಲ್ಲಿ ಹೊಸ ಜಾತಿಯನ್ನು ಕಂಡುಹಿಡಿದಿದ್ದಾರೆ.",
        "ಹಲವು ಭಾಷೆಗಳನ್ನು ಕಲಿಯುವುದು ಅರಿವಿನ ಸಾಮರ್ಥ್ಯ ಮತ್ತು ಪರಾನುಭೂತಿಯನ್ನು ಹೆಚ್ಚಿಸುತ್ತದೆ.",
        "ನಿಮ್ಮ ಸಮಯ ಮತ್ತು ಮಾರ್ಗದರ್ಶನಕ್ಕಾಗಿ ತುಂಬಾ ಧನ್ಯವಾದಗಳು."
    ]
}

directions = [
    ("en", "hi"),
    ("en", "kn"),
    ("hi", "en"),
    ("kn", "en"),
    ("hi", "kn"),
    ("kn", "hi"),
]

configs = [
    ("torch1b (beam 2)", "torch1b", "2"),
    ("torch1b (greedy)", "torch1b", "1"),
    ("onnx (int8 distilled)", "onnx", "1"),
]

benchmark_results = {}

print("==========================================================================")
print("BENCHMARK: 20 sentences per direction, 6 directions")
print("Comparing: torch1b beam 2, torch1b greedy, onnx int8 distilled")
print("==========================================================================\n")

for cfg_name, backend_mode, beam_val in configs:
    os.environ["TRANSLATOR_BACKEND"] = backend_mode
    os.environ["TR_BEAMS"] = beam_val
    tp.TRANSLATOR_BACKEND = backend_mode

    # Clear caches
    with tp._tr_lock:
        tp._text_cache.clear()
        tp._model_cache.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    print(f"\n>>> RUNNING CONFIG: {cfg_name} (backend={backend_mode}, beams={beam_val})")
    benchmark_results[cfg_name] = {}

    for src_iso, tgt_iso in directions:
        dir_key = f"{src_iso}->{tgt_iso}"
        sentences = test_sentences[src_iso]
        references = test_sentences[tgt_iso]
        latencies = []
        outputs = []

        # Preload model
        tp._get_model(src_iso, tgt_iso, target_backend=backend_mode)

        for text in sentences:
            t0 = time.perf_counter()
            out = tp.translate(text, src_iso, tgt_iso)
            elapsed_ms = (time.perf_counter() - t0) * 1000.0
            latencies.append(elapsed_ms)
            outputs.append(out)

        p50 = float(np.percentile(latencies, 50))
        p95 = float(np.percentile(latencies, 95))
        peak_vram_mb = (torch.cuda.max_memory_allocated() / (1024.0 * 1024.0)) if torch.cuda.is_available() else 0.0

        # Compute chrF++ score
        chrf = sacrebleu.corpus_chrf(outputs, [references], word_order=2)
        score_chrf = round(chrf.score, 2)

        benchmark_results[cfg_name][dir_key] = {
            "p50_ms": round(p50, 1),
            "p95_ms": round(p95, 1),
            "peak_vram_mb": round(peak_vram_mb, 1),
            "chrf2": score_chrf,
            "sample_src": sentences[0],
            "sample_out": outputs[0],
            "sample_ref": references[0],
        }
        print(f"  [{dir_key}] p50: {p50:.1f}ms | p95: {p95:.1f}ms | Peak VRAM: {peak_vram_mb:.1f}MB | chrF++: {score_chrf}")

# Print Summary Table
print("\n" + "=" * 95)
print(f"{'Direction':<10} | {'Config':<22} | {'p50 (ms)':<10} | {'p95 (ms)':<10} | {'VRAM (MB)':<10} | {'chrF++':<8}")
print("-" * 95)
for src_iso, tgt_iso in directions:
    dir_key = f"{src_iso}->{tgt_iso}"
    for cfg_name, _, _ in configs:
        res = benchmark_results[cfg_name][dir_key]
        print(f"{dir_key:<10} | {cfg_name:<22} | {res['p50_ms']:<10} | {res['p95_ms']:<10} | {res['peak_vram_mb']:<10} | {res['chrf2']:<8}")
    print("-" * 95)

# Eyeball Sample Outputs Side-by-Side
print("\n" + "=" * 95)
print("EYEBALL SAMPLE OUTPUTS SIDE-BY-SIDE")
print("=" * 95)
for src_iso, tgt_iso in directions:
    dir_key = f"{src_iso}->{tgt_iso}"
    sample_src = test_sentences[src_iso][0]
    sample_ref = test_sentences[tgt_iso][0]
    print(f"\nDirection: {dir_key}")
    print(f"  Source:    {sample_src}")
    print(f"  Reference: {sample_ref}")
    for cfg_name, _, _ in configs:
        out = benchmark_results[cfg_name][dir_key]["sample_out"]
        print(f"  {cfg_name:<22}: {out}")

# Recommendation Analysis
print("\n" + "=" * 95)
print("RECOMMENDATION DECISION")
print("=" * 95)
all_p95_torch1b_b2 = [benchmark_results["torch1b (beam 2)"][f"{s}->{t}"]["p95_ms"] for s, t in directions]
all_p95_torch1b_greedy = [benchmark_results["torch1b (greedy)"][f"{s}->{t}"]["p95_ms"] for s, t in directions]
max_p95_b2 = max(all_p95_torch1b_b2)
max_p95_greedy = max(all_p95_torch1b_greedy)

if max_p95_b2 <= 500.0:
    print(f"✅ RECOMMENDATION: Keep IndicTrans2 1B (beam 2).")
    print(f"   Reason: Worst-case p95 latency is {max_p95_b2:.1f}ms (<= 500ms target) with highest translation fidelity.")
elif max_p95_greedy <= 500.0:
    print(f"✅ RECOMMENDATION: Keep IndicTrans2 1B with greedy decoding (TR_BEAMS=1).")
    print(f"   Reason: Beam 2 p95 exceeded 500ms ({max_p95_b2:.1f}ms), but greedy p95 is {max_p95_greedy:.1f}ms (<= 500ms target)")
    print(f"   while retaining 1B parameter contextual translation superior to distilled ONNX models.")
else:
    print(f"⚠️ RECOMMENDATION: Automatic fallback to ONNX int8 distilled recommended for latency-critical paths.")
    print(f"   Reason: Both 1B configurations exceeded 500ms p95 on T4 GPU.")


From https://github.com/aananyaph/subtitlesssss
 * branch            main       -> FETCH_HEAD
Already up to date.
BENCHMARK: 20 sentences per direction, 6 directions
Comparing: torch1b beam 2, torch1b greedy, onnx int8 distilled


>>> RUNNING CONFIG: torch1b (beam 2) (backend=torch1b, beams=2)
[2026-11-07 07:21:19,542] [INFO] [IndicTrans2GPU] Loading model and tokenizer for ai4bharat/indictrans2-en-indic-1B...


INFO:translator_gpu:[IndicTrans2GPU] Loading model and tokenizer for ai4bharat/indictrans2-en-indic-1B...


[2026-11-07 07:21:19,737] [ERROR] [IndicTrans2GPU] Failed loading ai4bharat/indictrans2-en-indic-1B: IndicTransTokenizer has no attribute _special_tokens_map
Traceback (most recent call last):
  File "/content/subtitlesssss/translator_gpu.py", line 113, in __init__
    self.tokenizer = AutoTokenizer.from_pretrained(
                     ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        repo_id,
        ^^^^^^^^
        trust_remote_code=True,
        ^^^^^^^^^^^^^^^^^^^^^^^
        token=hf_token,
        ^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/transformers/models/auto/tokenization_auto.py", line 965, in from_pretrained
    return tokenizer_class.from_pretrained(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        pretrained_model_name_or_path, *inputs, trust_remote_code=trust_remote_code, **kwargs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/transformer

ERROR:translator_gpu:[IndicTrans2GPU] Failed loading ai4bharat/indictrans2-en-indic-1B: IndicTransTokenizer has no attribute _special_tokens_map
Traceback (most recent call last):
  File "/content/subtitlesssss/translator_gpu.py", line 113, in __init__
    self.tokenizer = AutoTokenizer.from_pretrained(
                     ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        repo_id,
        ^^^^^^^^
        trust_remote_code=True,
        ^^^^^^^^^^^^^^^^^^^^^^^
        token=hf_token,
        ^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/transformers/models/auto/tokenization_auto.py", line 965, in from_pretrained
    return tokenizer_class.from_pretrained(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        pretrained_model_name_or_path, *inputs, trust_remote_code=trust_remote_code, **kwargs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/transformers/tokenizatio

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

... (203 outputs left out)

INFO:translator_gpu:[IndicTrans2GPU] Loading model and tokenizer for ai4bharat/indictrans2-indic-en-1B...


[2026-11-07 07:21:56,249] [ERROR] [IndicTrans2GPU] Failed loading ai4bharat/indictrans2-indic-en-1B: IndicTransTokenizer has no attribute _special_tokens_map
Traceback (most recent call last):
  File "/content/subtitlesssss/translator_gpu.py", line 113, in __init__
    self.tokenizer = AutoTokenizer.from_pretrained(
                     ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        repo_id,
        ^^^^^^^^
        trust_remote_code=True,
        ^^^^^^^^^^^^^^^^^^^^^^^
        token=hf_token,
        ^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/transformers/models/auto/tokenization_auto.py", line 965, in from_pretrained
    return tokenizer_class.from_pretrained(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        pretrained_model_name_or_path, *inputs, trust_remote_code=trust_remote_code, **kwargs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/transformer

ERROR:translator_gpu:[IndicTrans2GPU] Failed loading ai4bharat/indictrans2-indic-en-1B: IndicTransTokenizer has no attribute _special_tokens_map
Traceback (most recent call last):
  File "/content/subtitlesssss/translator_gpu.py", line 113, in __init__
    self.tokenizer = AutoTokenizer.from_pretrained(
                     ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        repo_id,
        ^^^^^^^^
        trust_remote_code=True,
        ^^^^^^^^^^^^^^^^^^^^^^^
        token=hf_token,
        ^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/transformers/models/auto/tokenization_auto.py", line 965, in from_pretrained
    return tokenizer_class.from_pretrained(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        pretrained_model_name_or_path, *inputs, trust_remote_code=trust_remote_code, **kwargs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/transformers/tokenizatio

KeyboardInterrupt: 

In [ ]:
# 8. Start bridge (8766), wait, start proxy (8765), and verify listening ports
import os
import time
import subprocess
# Clean previous processes
os.system("fuser -k -9 8765/tcp 8766/tcp 2>/dev/null || true")
os.system("pkill -9 -f 'backend/app_bridge.py' || true")
os.system("pkill -9 -f 'translation_proxy.py' || true")
time.sleep(2)
# Launch bridge (8766)
bridge_log = open("/content/subtitlesssss/server.log", "w")
bridge_proc = subprocess.Popen(
    ["python", "backend/app_bridge.py", "--host", "127.0.0.1", "--model", "distil-small.en", "--port", "8766"],
    cwd="/content/subtitlesssss",
    env=dict(os.environ, QT_QPA_PLATFORM="offscreen"),
    stdout=bridge_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)
print(f"Launched app_bridge.py (PID: {bridge_proc.pid}). Waiting 15s...", flush=True)
time.sleep(15)
# Launch proxy (8765)
proxy_log = open("/content/subtitlesssss/proxy.log", "w")
proxy_proc = subprocess.Popen(
    ["python", "translation_proxy.py"],
    cwd="/content/subtitlesssss",
    env=dict(os.environ, PROXY_PORT="8765", UPSTREAM_URL="ws://127.0.0.1:8766", TRANSLATOR_BACKEND="torch1b"),
    stdout=proxy_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)
print(f"Launched translation_proxy.py (PID: {proxy_proc.pid}). Waiting 5s...", flush=True)
time.sleep(5)
print("\n--- Active Listening Ports ---", flush=True)
os.system("ss -ltnp | grep -E '8765|8766'")
print("\n--- GPU Memory Usage ---", flush=True)
os.system("nvidia-smi --query-gpu=memory.used,memory.total,memory.free --format=csv")

In [ ]:
# 9. Start pyngrok tunnel to proxy port 8765
import os
import time
from pyngrok import conf, ngrok

# 1. Load NGROK_AUTHTOKEN from Colab Secrets, environment, or prompt
token = None
try:
    from google.colab import userdata
    token = userdata.get('NGROK_AUTHTOKEN')
    if token:
        print("✅ Loaded NGROK_AUTHTOKEN from Colab Secrets.")
except Exception:
    pass

if not token:
    token = os.environ.get('NGROK_AUTHTOKEN')

if not token:
    print("⚠️ NGROK_AUTHTOKEN was not found in Colab Secrets (the 🔑 icon on the left panel).")
    token = input("Paste your ngrok authtoken here: ").strip() or None

# 2. Configure authtoken
if token:
    os.environ["NGROK_AUTHTOKEN"] = token
    ngrok.set_auth_token(token)
    !ngrok config add-authtoken $token
    print("✅ Ngrok authtoken configured.")

# 3. Terminate any previous ngrok instances
!pkill -9 -f ngrok || true
try:
    ngrok.kill()
except Exception:
    pass
time.sleep(2)

# 4. Connect tunnel using pyngrok Python API
domain = "aspire-reluctant-obsession.ngrok-free.dev"
print(f"Connecting ngrok tunnel for port 8765 on '{domain}'...")
try:
    tunnel = ngrok.connect(addr="8765", url=f"https://{domain}")
    public_url = tunnel.public_url
    wss_url = public_url.replace("https://", "wss://").replace("http://", "ws://")
    print("\n========================================================")
    print("✅ NGROK IS ONLINE!")
    print("Public HTTPS URL:", public_url)
    print("Client WebSocket URL:", wss_url)
    print("========================================================\n")
except Exception as e:
    print(f"⚠️ Could not bind to domain {domain}: {e}")
    print("Connecting with standard free ngrok tunnel...")
    tunnel = ngrok.connect(addr="8765")
    public_url = tunnel.public_url
    wss_url = public_url.replace("https://", "wss://").replace("http://", "ws://")
    print("\n========================================================")
    print("✅ NGROK IS ONLINE (Generated Tunnel)!")
    print("Public HTTPS URL:", public_url)
    print("Client WebSocket URL:", wss_url)
    print("👉 Update this URL in your PC Client Control Panel!")
    print("========================================================\n")


In [6]:
# 10. STEP 5: Verify Translation Layer & Latency (GPU & Proxy)
import os
import sys
import time

sys.path.insert(0, "/content/subtitlesssss")
import translation_proxy as tp

print("\n--- Testing Direct GPU Translation through Proxy Layer ---")
test_cases = [
    ("hi", "en", "नमस्ते, आप आज कैसे हैं?"),
    ("en", "hi", "Welcome to real-time subtitle translation."),
    ("kn", "en", "ನಮಸ್ಕಾರ, ನೀವು ಇಂದು ಹೇಗಿದ್ದೀರಿ?"),
]

for src, tgt, sample in test_cases:
    t0 = time.perf_counter()
    out, backend, cache_hit = tp.translate_with_meta(sample, src, tgt)
    dt_ms = (time.perf_counter() - t0) * 1000
    print(f"[{src} -> {tgt}] ({backend}) in {dt_ms:.1f}ms")
    print(f"   Input:  {sample}")
    print(f"   Output: {out}\n")

print("========================================================")
print("✅ GPU TRANSLATION LAYER FULLY VERIFIED AND WORKING!")
print("========================================================")


In [ ]:
!grep "SUB \|translation layer\|new upstream\|error" /content/subtitlesssss/proxy.log | tail -n 25
